# Fine-tune BERT — Aspect Sentiment Classification (SemEval-2014 Laptop)

Task: aspect-term polarity classification — given a sentence and a known aspect term inside it,
predict `positive` / `negative` / `neutral` (the `conflict` label has been dropped from the dataset —
too few examples to learn reliably, see baseline notes). Same task as the TF-IDF + Logistic Regression
baseline (`notebooks/baseline_semeval_laptop_kaggle_run.ipynb`), so results are directly comparable.

- Model: `bert-base-uncased`, fine-tuned as a sequence-pair classifier (`[CLS] sentence [SEP] aspect_term [SEP]`),
  the standard BERT-SPC setup for ABSA — no need for the `$T$` span-marking trick the sparse TF-IDF baseline used,
  since the transformer can attend across the two segments directly.
- Dataset: SemEval-2014 Task 4, Laptop domain, already split into `train.xml` / `valid.xml` / `test.xml`
  by `scripts/split_dataset.py` (15% valid / 15% test, seed=42) — loaded directly here instead of re-splitting,
  so the split is identical across baseline and all improved models.
- On Kaggle: add the dataset [`dattm03/genai-dataset`](https://www.kaggle.com/datasets/dattm03/genai-dataset) as a notebook input.
It mirrors this repo's `data/` folder, so `data/processed/laptop/{train,valid,test}.xml` (produced by `scripts/split_dataset.py`
from the official `Laptop_Train_v2.xml`) is available under `/kaggle/input/genai-dataset/...`.
Also turn on a **GPU accelerator** (Settings → Accelerator → GPU T4 x2 or similar) — fine-tuning on CPU is impractically slow.


In [ ]:
import glob
import json
from dataclasses import dataclass, field
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEEDS = [42, 43, 44]


## 1. Locate the data

Works both on Kaggle (`/kaggle/input/...`) and locally (repo `data/processed/laptop/`). Searches for a
directory that contains all three of `train.xml`, `valid.xml`, `test.xml` together, to avoid matching
unrelated XML files elsewhere in the dataset (e.g. `data/raw/train/Laptop_Train_v2.xml`).

In [ ]:
def find_processed_dir():
    search_roots = ["/kaggle/input", "../data/processed/laptop", "data/processed/laptop"]
    for root in search_roots:
        for valid_path in glob.glob(f"{root}/**/valid.xml", recursive=True) + glob.glob(f"{root}/valid.xml"):
            d = Path(valid_path).parent
            if (d / "train.xml").exists() and (d / "test.xml").exists():
                return d
    raise FileNotFoundError(
        "train.xml/valid.xml/test.xml not found. On Kaggle: add the 'dattm03/genai-dataset' dataset as "
        "notebook input. Locally: run scripts/split_dataset.py to produce data/processed/laptop/."
    )


DATA_DIR = find_processed_dir()
print("Using data dir:", DATA_DIR)


## 2. Parse the SemEval XML

Same logic as `src/data/semeval_loader.py` in the repo, inlined here so the notebook is self-contained on Kaggle.

In [ ]:
@dataclass
class AspectTerm:
    term: str
    polarity: str
    start: int
    end: int


@dataclass
class Sentence:
    sentence_id: str
    text: str
    aspect_terms: list = field(default_factory=list)


def load_semeval_xml(path):
    """Load a SemEval-2014 Task 4 XML file. Aspect terms without a `polarity`
    attribute (unlabeled blind test data) are skipped; the sentence is still returned."""
    root = ET.parse(path).getroot()
    sentences = []
    for sent_el in root.findall("sentence"):
        text = sent_el.findtext("text") or ""
        aspect_terms = []
        for term_el in sent_el.findall("./aspectTerms/aspectTerm"):
            polarity = term_el.get("polarity")
            if polarity is None:
                continue
            aspect_terms.append(
                AspectTerm(
                    term=term_el.get("term", ""),
                    polarity=polarity,
                    start=int(term_el.get("from", -1)),
                    end=int(term_el.get("to", -1)),
                )
            )
        sentences.append(
            Sentence(sentence_id=sent_el.get("id", ""), text=text, aspect_terms=aspect_terms)
        )
    return sentences


train_sentences = load_semeval_xml(DATA_DIR / "train.xml")
valid_sentences = load_semeval_xml(DATA_DIR / "valid.xml")
test_sentences = load_semeval_xml(DATA_DIR / "test.xml")

for name, sents in [("train", train_sentences), ("valid", valid_sentences), ("test", test_sentences)]:
    n_terms = sum(len(s.aspect_terms) for s in sents)
    print(f"{name:>5}: {len(sents):4d} sentences, {n_terms:4d} aspect terms")


## 3. Build (sentence, aspect, polarity) examples

Flatten each sentence into one example per labeled aspect term — same flattening as
`src/data/preprocess.py`, minus the `$T$` marking (not needed for sentence-pair input).

In [ ]:
LABELS = ["negative", "neutral", "positive"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}


def build_examples(sentences):
    texts, aspects, labels = [], [], []
    for sent in sentences:
        for term in sent.aspect_terms:
            if term.polarity not in label2id:
                continue
            texts.append(sent.text)
            aspects.append(term.term)
            labels.append(label2id[term.polarity])
    return texts, aspects, labels


train_texts, train_aspects, train_labels = build_examples(train_sentences)
valid_texts, valid_aspects, valid_labels = build_examples(valid_sentences)
test_texts, test_aspects, test_labels = build_examples(test_sentences)

print(f"Train examples: {len(train_labels)} | Valid examples: {len(valid_labels)} | Test examples: {len(test_labels)}")


## 4. Tokenize & build PyTorch datasets

Model: `bert-base-uncased`. Input is a sentence pair `(sentence, aspect_term)` so the tokenizer
produces `[CLS] sentence [SEP] aspect_term [SEP]` with the right `token_type_ids`.

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(texts, aspects):
    return tokenizer(
        texts,
        aspects,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )


class AbsaDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = AbsaDataset(encode(train_texts, train_aspects), train_labels)
valid_dataset = AbsaDataset(encode(valid_texts, valid_aspects), valid_labels)
test_dataset = AbsaDataset(encode(test_texts, test_aspects), test_labels)


## 5. Shared training components

- **Class-weighted loss**: the baseline showed the label distribution is skewed
  (`positive`/`negative` dominate, `neutral` is the minority class) and `neutral` suffered most.
  We pass per-class weights (inverse frequency, computed from the train split) into the
  cross-entropy loss via a `Trainer` subclass, mirroring `class_weight="balanced"` used in the
  baseline's Logistic Regression.
- **`compute_metrics`** reports accuracy + macro-F1 on every eval call (used both for model
  selection and early stopping — macro-F1, not accuracy, since accuracy is dominated by the
  majority classes).

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(LABELS)),
    y=np.array(train_labels),
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", dict(zip(LABELS, class_weights.tolist())))


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, len(LABELS)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }


## 6. Train across 3 seeds (42, 43, 44), 20 epochs each

20 epochs is a lot for ~2000 train examples, so two choices keep it from overfitting instead of
just memorizing the train split:

- **Learning rate — fixed at `2e-05`**: a hand-picked, balanced value (not a dynamic
  schedule) — the standard fine-tuning learning rate recommended for BERT-base (the original BERT paper's 2e-5-5e-5 sweet spot), left unchanged even with 20 epochs available.
- **Early stopping**: `EarlyStoppingCallback(early_stopping_patience=3)` — training stops once
  valid macro-F1 hasn't improved for 3 consecutive epochs, and `load_best_model_at_end=True`
  restores the best checkpoint, so the extra epoch budget is only spent if it actually helps.

Each seed gets a fresh model (`from_pretrained`) and its own output dir, so no state leaks across
runs. The model is re-evaluated on the **test** split after each seed's training finishes, and the
best-by-macro-F1 seed's model is kept on disk (the others are discarded to save space/GPU memory).

In [ ]:
out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("results")
out_dir.mkdir(parents=True, exist_ok=True)
BASE_OUTPUT_DIR = "/kaggle/working/bert-absa" if Path("/kaggle/working").exists() else "results/bert-absa"

seed_results = []
best_macro_f1 = -1.0
best_seed = None

for seed in SEEDS:
    print(f"\n===== Seed {seed} =====")
    set_seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    common_args = dict(
        output_dir=f"{BASE_OUTPUT_DIR}-seed{seed}",
        num_train_epochs=20,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-05,
        weight_decay=0.01,
        logging_steps=50,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        seed=seed,
        report_to="none",
    )
    try:
        training_args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **common_args)
    except TypeError:
        # older transformers versions use `evaluation_strategy` instead of `eval_strategy`
        training_args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **common_args)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()

    test_output = trainer.predict(test_dataset)
    test_preds = np.argmax(test_output.predictions, axis=-1)
    accuracy = accuracy_score(test_labels, test_preds)
    macro_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
    report = classification_report(
        test_labels, test_preds, labels=list(id2label.keys()), target_names=LABELS,
        zero_division=0, output_dict=True,
    )
    print(f"Seed {seed} -> Accuracy {accuracy:.4f} | Macro-F1 {macro_f1:.4f}")

    seed_results.append({"seed": seed, "accuracy": accuracy, "macro_f1": macro_f1, "classification_report": report})

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_seed = seed
        model_dir = out_dir / "bert-absa-model"
        trainer.save_model(str(model_dir))
        tokenizer.save_pretrained(str(model_dir))

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nBest seed: {best_seed} (Macro-F1 {best_macro_f1:.4f}) -> saved to {out_dir / 'bert-absa-model'}")


## 7. Aggregate results across seeds

Mean ± sample std (ddof=1) of accuracy and macro-F1 over the 3 seeds, plus a per-label
precision/recall/F1 breakdown (support is fixed — same test split every seed).

In [ ]:
accuracies = [r["accuracy"] for r in seed_results]
macro_f1s = [r["macro_f1"] for r in seed_results]

print(f"Accuracy: {np.mean(accuracies):.4f} +/- {np.std(accuracies, ddof=1):.4f}  (per seed: {[round(a, 4) for a in accuracies]})")
print(f"Macro-F1: {np.mean(macro_f1s):.4f} +/- {np.std(macro_f1s, ddof=1):.4f}  (per seed: {[round(m, 4) for m in macro_f1s]})")


def summarize_per_label(results, labels):
    rows = []
    for label in labels:
        f1s = [r["classification_report"][label]["f1-score"] for r in results]
        precisions = [r["classification_report"][label]["precision"] for r in results]
        recalls = [r["classification_report"][label]["recall"] for r in results]
        rows.append({
            "label": label,
            "precision_mean": np.mean(precisions),
            "recall_mean": np.mean(recalls),
            "f1_mean": np.mean(f1s),
            "f1_std": np.std(f1s, ddof=1),
            "support": results[0]["classification_report"][label]["support"],
        })
    return pd.DataFrame(rows)


per_label_df = summarize_per_label(seed_results, LABELS)
print()
print(per_label_df.to_string(index=False))


## 8. Save aggregated metrics

In [ ]:
metrics_path = out_dir / "bert_metrics.json"
metrics_path.write_text(json.dumps({
    "model_name": MODEL_NAME,
    "seeds": SEEDS,
    "best_seed": best_seed,
    "mean_accuracy": float(np.mean(accuracies)),
    "std_accuracy": float(np.std(accuracies, ddof=1)),
    "mean_macro_f1": float(np.mean(macro_f1s)),
    "std_macro_f1": float(np.std(macro_f1s, ddof=1)),
    "train_examples": len(train_labels),
    "valid_examples": len(valid_labels),
    "test_examples": len(test_labels),
    "per_seed_results": seed_results,
}, indent=2))
print(f"Saved metrics to {metrics_path}")
print(f"Best model saved to {out_dir / 'bert-absa-model'}")


## Next steps

- Copy the mean±std `Accuracy` / `Macro-F1` back into `plans/project-plan.md` (Tuần 4) and compare
  against the TF-IDF + Logistic Regression baseline (Accuracy 0.6211 / Macro-F1 0.4266) — note that
  baseline number was measured on the old 4-class split (with `conflict`); re-run the baseline on the
  current 3-class data for a strictly apples-to-apples comparison. Expect the biggest gains on `neutral`.
- Compare against the sibling notebook (DistilBERT fine-tune) to
  pick the better model for the aspect-statistics + report-generation steps later in Tuần 4.
- Reuse `bert-absa-model/` (the best-of-3-seeds checkpoint) to score the Amazon Reviews demo
  set once aspect extraction is available.
